In [ ]:
import sys
import subprocess

def ensure_installed(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

ensure_installed(['transformers', 'datasets', 'torch', 'scikit-learn', 'pandas', 'numpy'])


In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report

if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    pipeline_device = torch.device('mps')
else:
    device = 'cpu'
    pipeline_device = -1

print({'selected_device': device})


In [ ]:
class_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

goemotions = load_dataset('go_emotions', 'raw', split='test')

goemotions_to_canonical = {
    'sadness': 'sadness',
    'grief': 'sadness',
    'disappointment': 'sadness',
    'remorse': 'sadness',
    'embarrassment': 'sadness',
    'joy': 'joy',
    'amusement': 'joy',
    'excitement': 'joy',
    'optimism': 'joy',
    'pride': 'joy',
    'relief': 'joy',
    'gratitude': 'joy',
    'approval': 'joy',
    'admiration': 'joy',
    'caring': 'love',
    'desire': 'love',
    'love': 'love',
    'anger': 'anger',
    'annoyance': 'anger',
    'disapproval': 'anger',
    'disgust': 'anger',
    'fear': 'fear',
    'nervousness': 'fear',
    'surprise': 'surprise',
    'realization': 'surprise',
    'confusion': 'surprise'
}

selected_rows = []
for row in goemotions:
    labels = row['labels']
    if len(labels) != 1:
        continue
    raw_label = goemotions.features['labels'].feature.int2str(labels[0])
    if raw_label in goemotions_to_canonical:
        selected_rows.append({
            'text': row['text'],
            'goemotions_label': raw_label,
            'canonical_label': goemotions_to_canonical[raw_label]
        })
    if len(selected_rows) >= 300:
        break

subset_df = pd.DataFrame(selected_rows)
label_to_id = {name: i for i, name in enumerate(class_names)}
true_ids = [label_to_id[x] for x in subset_df['canonical_label']]

print({
    'dataset': 'go_emotions',
    'config': 'raw',
    'split': 'test',
    'subset_size': len(subset_df),
    'class_distribution': subset_df['canonical_label'].value_counts().to_dict()
})
print(subset_df.head(10).to_dict(orient='records'))


In [ ]:
model_name = 'j-hartmann/emotion-english-distilroberta-base'

model_label_to_canonical = {
    'sadness': 'sadness',
    'joy': 'joy',
    'love': 'love',
    'anger': 'anger',
    'fear': 'fear',
    'surprise': 'surprise'
}

clf = pipeline(
    task='text-classification',
    model=model_name,
    tokenizer=model_name,
    device=pipeline_device,
    truncation=True
)

print({
    'model_name': model_name,
    'task': 'text-classification',
    'model_output_labels': sorted(model_label_to_canonical.keys())
})


In [ ]:
texts = subset_df['text'].tolist()
batch_size = 32

pred_outputs = clf(texts, batch_size=batch_size)

raw_pred_labels = [output['label'].lower() for output in pred_outputs]
pred_labels = [model_label_to_canonical[label] for label in raw_pred_labels]
pred_ids = [label_to_id[x] for x in pred_labels]

results_df = subset_df.copy()
results_df['model_raw_label'] = raw_pred_labels
results_df['predicted_label'] = pred_labels

print(results_df.head(10).to_dict(orient='records'))


In [ ]:
accuracy = accuracy_score(true_ids, pred_ids)
report = classification_report(true_ids, pred_ids, target_names=class_names, digits=4)

print({
    'model_name': model_name,
    'dataset': 'go_emotions',
    'config': 'raw',
    'split': 'test',
    'subset_size': len(results_df),
    'device': device,
    'accuracy': round(float(accuracy), 6)
})
print(report)


In [ ]:
sample_n = 12
print(results_df[['text', 'goemotions_label', 'canonical_label', 'model_raw_label', 'predicted_label']].head(sample_n).to_string(index=False))
